Imports

In [1]:
import asyncio
import threading
import time

Task 1

In [2]:
def run_sync_tasks():
    start = time.time()
    for i in range(1, 9):
        print(f"Task {i} start")
        time.sleep(1)
        print(f"Task {i} finish")
    print("Total time:", round(time.time() - start, 2))

run_sync_tasks()

Task 1 start
Task 1 finish
Task 2 start
Task 2 finish
Task 3 start
Task 3 finish
Task 4 start
Task 4 finish
Task 5 start
Task 5 finish
Task 6 start
Task 6 finish
Task 7 start
Task 7 finish
Task 8 start
Task 8 finish
Total time: 8.01


Task 2

In [3]:
async def task(n):
    print(f"Task {n} start")
    await asyncio.sleep(1)
    print(f"Task {n} finish")

async def run_async_sequential():
    start = time.time()
    for i in range(1, 9):
        await task(i)
    print("Total time:", round(time.time() - start, 2))

await run_async_sequential()

Task 1 start
Task 1 finish
Task 2 start
Task 2 finish
Task 3 start
Task 3 finish
Task 4 start
Task 4 finish
Task 5 start
Task 5 finish
Task 6 start
Task 6 finish
Task 7 start
Task 7 finish
Task 8 start
Task 8 finish
Total time: 8.01


Task 3

In [4]:
async def run_async_concurrent():
    start = time.time()
    tasks = [task(i) for i in range(1, 9)]
    await asyncio.gather(*tasks)
    print("Total time:", round(time.time() - start, 2))

await run_async_concurrent()

Task 1 start
Task 2 start
Task 3 start
Task 4 start
Task 5 start
Task 6 start
Task 7 start
Task 8 start
Task 1 finish
Task 3 finish
Task 7 finish
Task 6 finish
Task 8 finish
Task 5 finish
Task 2 finish
Task 4 finish
Total time: 1.0


Task 4

In [5]:
async def ticker():
    for i in range(10):
        print("tick", i)
        await asyncio.sleep(0.5)

def heavy_cpu_function():
    total = 0
    for i in range(30_000_000):
        total += i * i
    return total

async def main_blocking():
    t = asyncio.create_task(ticker())
    await asyncio.sleep(1.2)
    print("heavy start")
    heavy_cpu_function()
    print("heavy done")
    await t

await main_blocking()

tick 0
tick 1
tick 2
heavy start
heavy done
tick 3
tick 4
tick 5
tick 6
tick 7
tick 8
tick 9


Task 5

In [6]:
async def cpu_with_yield():
    total = 0
    for i in range(5_000_000):
        total += i * i
        if i % 200_000 == 0:
            await asyncio.sleep(0)
    return total

async def main_yield():
    await asyncio.gather(ticker(), cpu_with_yield())

await main_yield()

tick 0
tick 1
tick 2
tick 3
tick 4
tick 5
tick 6
tick 7
tick 8
tick 9


Task 6

In [7]:
async def main_thread():
    t = asyncio.create_task(ticker())
    await asyncio.sleep(1.2)
    print("heavy start")
    await asyncio.to_thread(heavy_cpu_function)
    print("heavy done")
    await t

await main_thread()

tick 0
tick 1
tick 2
heavy start
tick 3
tick 4
heavy done
tick 5
tick 6
tick 7
tick 8
tick 9


Task 7

In [8]:
shared_counter = 0
lock = threading.Lock()

def add_no_lock():
    global shared_counter
    for _ in range(10_000):
        temp = shared_counter
        time.sleep(0)
        shared_counter = temp + 1

def add_with_lock():
    global shared_counter
    for _ in range(10_000):
        with lock:
            temp = shared_counter
            time.sleep(0)
            shared_counter = temp + 1

def run_threads(func):
    global shared_counter
    shared_counter = 0
    threads = [threading.Thread(target=func) for _ in range(10)]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    return shared_counter

print("Without lock:", run_threads(add_no_lock))
print("With lock:", run_threads(add_with_lock))
print("Expected:", 10 * 10_000)

Without lock: 10817
With lock: 100000
Expected: 100000


Task 8

In [9]:
counter = 0
counter_lock = threading.Lock()
def cpu_work(n):
    global counter
    total = 0
    for i in range(2_000_000):
        total += i * i
    for _ in range(10_000):
        with counter_lock:
            counter += 1
    print(f"Task {n} done in {threading.current_thread().name}")

async def worker(n):
    print(f"Task {n} requesting cpu work")
    await asyncio.to_thread(cpu_work, n)

async def main_hybrid():
    await asyncio.gather(*(worker(i) for i in range(1, 6)))
    print("Final counter:", counter)
    print("Expected:", 5 * 10_000)

await main_hybrid()

Task 1 requesting cpu work
Task 2 requesting cpu work
Task 3 requesting cpu work
Task 4 requesting cpu work
Task 5 requesting cpu work
Task 2 done in asyncio_1
Task 1 done in asyncio_0
Task 3 done in asyncio_2
Task 5 done in asyncio_4
Task 4 done in asyncio_3
Final counter: 50000
Expected: 50000
